# Bloque V — Series temporales y forecasting

**Duración estimada:** 3 horas  
**Dataset:** `../data/demanda_diaria_mayo_2026.csv`

## Objetivo de aprendizaje

El alumnado aprenderá a analizar una serie temporal, crear variables temporales, construir modelos baseline y entrenar un modelo de forecasting con Machine Learning.

## Agenda de 3 horas

| Tiempo | Actividad |
|---:|---|
| 0:00–0:25 | Qué es una serie temporal |
| 0:25–0:55 | Indexación y visualización |
| 0:55–1:25 | Tendencia, estacionalidad y ruido |
| 1:25–1:35 | Pausa |
| 1:35–2:05 | Baselines y medias móviles |
| 2:05–2:35 | Forecasting con Machine Learning |
| 2:35–3:00 | Evaluación e interpretación |

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    STATSMODELS_OK = True
except Exception:
    STATSMODELS_OK = False

## 1. Carga e indexación temporal

En series temporales el orden de los datos es parte del problema. Por eso convertimos la fecha y la usamos como índice.

In [ ]:
df = pd.read_csv("../data/demanda_diaria_mayo_2026.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").set_index("fecha")
df.head()

## 2. Visualización inicial

El primer paso es representar la serie para detectar tendencia, estacionalidad, ruido y posibles anomalías.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df.index, df["demanda"])
plt.title("Evolución diaria de la demanda")
plt.xlabel("Fecha")
plt.ylabel("Demanda")
plt.show()

## 3. Componentes temporales

Una serie temporal puede contener:

- tendencia,
- estacionalidad,
- ruido,
- cambios de nivel,
- picos o anomalías.

Si `statsmodels` está disponible, realizamos una descomposición semanal.

In [ ]:
if STATSMODELS_OK:
    descomposicion = seasonal_decompose(df["demanda"], model="additive", period=7)
    descomposicion.plot()
    plt.show()
else:
    print("statsmodels no está disponible en este entorno.")

## 4. Ingeniería de variables temporales

Para aplicar Machine Learning a series temporales, transformamos el tiempo en variables predictoras:

- día de la semana,
- mes,
- variables lag,
- medias móviles,
- variables externas como promoción, festivo o temperatura.

In [ ]:
df_features = df.copy()

df_features["dia_semana"] = df_features.index.dayofweek
df_features["mes"] = df_features.index.month
df_features["trimestre"] = df_features.index.quarter
df_features["lag_1"] = df_features["demanda"].shift(1)
df_features["lag_7"] = df_features["demanda"].shift(7)
df_features["lag_14"] = df_features["demanda"].shift(14)
df_features["media_7"] = df_features["demanda"].rolling(window=7).mean()
df_features["media_14"] = df_features["demanda"].rolling(window=14).mean()

df_features.head(15)

In [ ]:
df_modelo = df_features.dropna()
df_modelo.head()

## 5. Modelos baseline

Un modelo avanzado debe compararse con una referencia sencilla.

Baseline 1: predicción ingenua con el último valor conocido.  
Baseline 2: media móvil semanal.

In [ ]:
df_eval = df_modelo.copy()
df_eval["baseline_naive"] = df_eval["lag_1"]
df_eval["baseline_media_7"] = df_eval["media_7"]

df_eval[["demanda", "baseline_naive", "baseline_media_7"]].head()

## 6. Separación temporal train/test

En series temporales no mezclamos aleatoriamente. Entrenamos con el pasado y probamos con el futuro.

In [ ]:
horizonte_test = 20

train = df_modelo.iloc[:-horizonte_test]
test = df_modelo.iloc[-horizonte_test:]

print("Train:", train.index.min(), "->", train.index.max())
print("Test:", test.index.min(), "->", test.index.max())

## 7. Evaluación de baselines

In [ ]:
def metricas_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

mae_naive, rmse_naive = metricas_forecast(test["demanda"], test["lag_1"])
mae_ma7, rmse_ma7 = metricas_forecast(test["demanda"], test["media_7"])

pd.DataFrame([
    {"modelo": "Naive lag_1", "MAE": mae_naive, "RMSE": rmse_naive},
    {"modelo": "Media móvil 7", "MAE": mae_ma7, "RMSE": rmse_ma7},
])

## 8. Forecasting con Random Forest

Usamos variables temporales, lags, medias móviles y variables externas.

In [ ]:
features = [
    "dia_semana", "mes", "trimestre",
    "lag_1", "lag_7", "lag_14",
    "media_7", "media_14",
    "promocion", "festivo", "temperatura", "stock_disponible"
]

target = "demanda"

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

modelo_rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    min_samples_leaf=2
)

modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)

mae_rf, rmse_rf = metricas_forecast(y_test, pred_rf)

pd.DataFrame([
    {"modelo": "Naive lag_1", "MAE": mae_naive, "RMSE": rmse_naive},
    {"modelo": "Media móvil 7", "MAE": mae_ma7, "RMSE": rmse_ma7},
    {"modelo": "Random Forest", "MAE": mae_rf, "RMSE": rmse_rf},
]).sort_values("RMSE")

## 9. Visualización de predicciones

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(test.index, y_test, label="Real", marker="o", linewidth=2)
plt.plot(test.index, pred_rf, label="Random Forest", marker="s", linewidth=2)
plt.plot(test.index, test["lag_1"], label="Baseline Naive", linestyle="--", linewidth=1)
plt.plot(test.index, test["media_7"], label="Baseline Media 7", linestyle="--", linewidth=1)
plt.title("Comparativa de predicciones")
plt.xlabel("Fecha")
plt.ylabel("Demanda")
plt.legend()
plt.show()

## 10. Importancia de características

In [ ]:
importancias = pd.DataFrame({
    "caracteristica": features,
    "importancia": modelo_rf.feature_importances_
}).sort_values("importancia", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(importancias["caracteristica"], importancias["importancia"])
plt.title("Importancia de características en Random Forest")
plt.xlabel("Importancia")
plt.show()

importancias

## 11. Conclusiones y mejoras futuras

### Conclusiones:
1. **Rendimiento del modelo**: El Random Forest captura mejor la estacionalidad que los baselines simples.
2. **Variables clave**: Los lags (valores pasados) y la media móvil son predictores más importantes que variables externas.
3. **Validación temporal**: Es crítico usar separación temporal (no aleatoria) en series temporales.

### Mejoras futuras:
- Ajustar hiperparámetros del Random Forest
- Probar modelos especializados: ARIMA, Prophet, LSTM
- Validación cruzada temporal (rolling window)
- Incluir más variables externas (festivos, eventos especiales)